# Numerical IK Math (Updated to Current Code)

This note matches the **current math** used in the solver (`kinematics.py`, `planner.py`, `constraints.py`, `main.py`, `create-json.py`).

---

## 1) State, target, and units

### Joint vector
$$
q = [q_1,q_2,q_3,q_4,q_5,q_6]^\top \in \mathbb{R}^6
$$

IK is solved in **radians**.

### Cartesian pose target format
Each target is a 6-vector:
$$
x_{\text{target}} = [p_x,p_y,p_z,r_x,r_y,r_z]^\top
$$

- first 3 entries: desired position (mm)
- last 3 entries: **rotation vector** (axis-angle packed as a vector)

So orientation is **not Euler angles**.

---

## 2) Homogeneous transforms (used in `kinematics.py`)

A rigid transform is built as

$$
T(R,t)=
\begin{bmatrix}
R & t\\
0 & 1
\end{bmatrix}
$$

where $R\in\mathbb{R}^{3\times3}$ and $t\in\mathbb{R}^3$, using standard $R_x(a), R_y(a), R_z(a)$.

---

## 3) Forward kinematics (current transform chain)

The current FK is an **explicit transform chain** (not the older DH derivation notes):

1. **Base**
$$
T_{\text{base}} = T(R_z(q_1), [0,0,99.1+63.4]^\top)
$$

2. **Shoulder**
$$
T_{\text{shoulder}} = T(R_y(q_2), [0,-137.8,0]^\top)
$$

3. **Elbow**
$$
T_{\text{elbow}} = T(R_y(q_3), [0,131.8,425]^\top)
$$

4. **Forearm**
$$
T_{\text{forearm}} = T(R_y(q_4), [0,-126.7,392.2]^\top)
$$

5. **Wrist**
$$
T_{\text{wrist}} = T(R_z(q_5), [0,0,99.7]^\top)
$$

6. **End effector**
$$
T_{\text{ee}} = T(R_x(q_6), [98.9,0,0]^\top)
$$

Final pose:
$$
T_{\text{EE}}(q)=
T_0\,
T_{\text{base}}\,
T_{\text{shoulder}}\,
T_{\text{elbow}}\,
T_{\text{forearm}}\,
T_{\text{wrist}}\,
T_{\text{ee}},
\quad T_0=I_4
$$

---

## 4) Rotation vector math (`vectorToR`, `RToVector`)

A rotation vector $r\in\mathbb{R}^3$ stores axis-angle as:

$$
r = \theta \hat{k}
$$

where:
- $\theta = \|r\|$ is rotation angle
- $\hat{k}$ is the unit axis

### Rotation vector $\to$ rotation matrix (Rodrigues)
If $K=[\hat{k}]_\times$, then:

$$
R = I + \sin\theta\,K + (1-\cos\theta)K^2
$$

### Rotation matrix $\to$ rotation vector
$$
\theta = \cos^{-1}\!\left(\frac{\operatorname{tr}(R)-1}{2}\right)
$$

Then extract the axis from the skew-symmetric part and return $r=\theta\hat{k}$.

---

## 5) Pose error used by IK (`poseError`)

From FK, current pose is:
$$
T(q)=\begin{bmatrix}R_{\text{cur}} & p_{\text{cur}}\\0&1\end{bmatrix}
$$

Given target $x_{\text{target}}=[p_{\text{des}}, r_{\text{des}}]$:
- $R_{\text{des}}=\text{vectorToR}(r_{\text{des}})$

### Position error
$$
e_{\text{pos}} = p_{\text{des}} - p_{\text{cur}}
$$

### Orientation error
$$
R_{\text{err}} = R_{\text{des}}R_{\text{cur}}^\top
$$
$$
e_{\text{rot}} = \text{RToVector}(R_{\text{err}})
$$

### Weighted 6D residual
$$
e(q)=
\begin{bmatrix}
w_{\text{pos}}\,e_{\text{pos}}\\
w_{\text{rot}}\,e_{\text{rot}}
\end{bmatrix}
\in \mathbb{R}^6
$$

Defaults in the current code:
- $w_{\text{pos}}=1.0$
- $w_{\text{rot}}=0.005$

So orientation error is intentionally weighted much lower than position error.

---

## 6) Jacobian by finite differences (`Jacobianfd`)

The Jacobian is **numerical** (not analytic):

$$
J(q)=\frac{\partial e}{\partial q}\in\mathbb{R}^{6\times 6}
$$

Each column is approximated by finite differences:

$$
J_{:,i}\approx\frac{e(q+\varepsilon e_i)-e(q)}{\varepsilon}
$$

where:
- $e_i$ is the $i$-th basis vector
- $\varepsilon = 10^{-6}$ in code

---

## 7) Damped least-squares IK with smoothing (`solvePoseIK`)

At each iteration, compute residual $e$ and Jacobian $J$, then solve:

$$
A\Delta q = b
$$

with

$$
A = J^\top J + (\lambda+\mu)I
$$

$$
b = -J^\top e - \mu(q-q_{\text{prev}})
$$

Then update:

$$
q \leftarrow q + \alpha \Delta q
$$

where:
- $\lambda$ = `damping`
- $\mu$ = `smoothw`
- $\alpha$ = `stepScale`

Finally, clamp to joint limits.

### Interpretation
- $J^\top J$: least-squares normal equations
- $+\lambda I$: damping (stability / singularity robustness)
- smoothing term: encourages continuity with previous waypoint solution

---

## 8) Joint limits and clamping (`clampQ`)

For lower and upper bounds $q^{\min}, q^{\max}$, clamping is:

$$
q_{\text{cmd}}=\min(\max(q,q^{\min}),q^{\max})
$$

The clamp function also reports:
- whether any limit was hit
- which joint indices violated limits

---

## 9) Reach checks (`checkReach`)

Before IK, the target position is pre-filtered using simple checks:
- radial max: $\|p\|\le r_{\max}$
- radial min: $\|p\|\ge r_{\min}$
- per-coordinate minimum magnitude checks: $|x|,|y|,|z|\ge r_{\min\_reach}$

These are quick sanity checks, not a full exact workspace test.

---

## 10) Trajectory generation (`generateTrajectoryPose`)

Given pose targets $x_0,\dots,x_{N-1}$, IK is solved **sequentially**:

1. initialize $q=q_{\text{start}}$
2. for each target $x_i$:
   - solve IK from current $q$
   - use $q_{\text{prev}}$ in the smoothing term
   - store the solved configuration
3. warm-start the next target from the current solution

This warm-started sequence helps keep motion smooth and improves convergence.

---

## 11) Target interpolation and JSON export (`create-json.py`, `main.py`)

Targets are built by linear interpolation in pose-vector space:

$$
\text{targets}=\text{linspace}(\text{start},\text{goal},N)
$$

where each target is $[x,y,z,r_x,r_y,r_z]$.

After IK, joint trajectories are exported in **degrees** for playback:

$$
q_{\deg}=\operatorname{rad2deg}(q)
$$

---

## 12) Jacobian conditioning diagnostic (from `main.py`)

SVD of the final Jacobian:
$$
J=U\Sigma V^\top
$$

Condition number:
$$
\kappa(J)=\frac{\sigma_{\max}}{\sigma_{\min}}
$$

A large $\kappa(J)$ indicates an ill-conditioned / near-singular posture.

---

## 13) Summary (current math)

The current solver does:

1. explicit-transform-chain FK  
2. weighted pose residual (position + rotation-vector error)  
3. finite-difference Jacobian  
4. damped least-squares IK with smoothing toward previous waypoint  
5. joint clamping  
6. sequential IK over interpolated Cartesian pose targets